# Colab 07 — ¿Tres instrumentos distintos miden la misma velocidad?

**Laboratorio 1 · Departamento de Física · FCEN-UBA**

Clase 7 — 23/09

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/charlyacha/Labo1-colabs/blob/main/07_Datos_reales_adquisicion_y_derivadas.ipynb)

Registraste el mismo móvil con photogate, con el sistema de adquisición y con Tracker. Tres archivos, tres formatos, tres incertezas. Hoy los leemos y decidimos si dan lo mismo.

**Al terminar vas a poder:** leer archivos reales sin pelearte con el formato, recortar el tramo útil de una serie temporal, y saber por qué conviene ajustar antes que derivar.

---

### Antes de tocar nada

Andá a **Archivo → Guardar una copia en Drive**. Vas a trabajar sobre tu copia:
lo que escribas acá sin copiar primero no se guarda en ningún lado.

Este cuaderno se recorre **de arriba hacia abajo**. Las celdas no son
independientes: cada una usa lo que definieron las anteriores. Si algo tira
`NameError`, casi siempre es porque salteaste una celda.

In [ ]:
import os

if not os.path.exists("lab1_utils.py"):
    !wget -q -O lab1_utils.py https://raw.githubusercontent.com/charlyacha/Labo1-colabs/main/lab1_utils.py

import numpy as np
import matplotlib.pyplot as plt
import lab1_utils as lab

lab.estilo_lab1()
print("Listo. numpy", np.__version__)

### 1. Los archivos reales vienen sucios

Ésta es la clase donde el análisis deja de ser un ejercicio. Un archivo que
sale de un instrumento tiene encabezados de largo variable, separadores
impredecibles, filas vacías y —el clásico que arruina más tardes de las que
parece— **coma decimal**, porque el programa estaba configurado en español.

La celda que sigue fabrica tres archivos con tres formatos distintos, uno
por instrumento, para que veas el problema completo.

In [ ]:
generador = np.random.default_rng(42)

a_real, v0_real, x0_real = 1.842, 0.115, 0.0203


def posicion(t):
    return x0_real + v0_real*t + 0.5*a_real*t**2


# --- Tracker: coma decimal, tabuladores, encabezado de metadatos, NaN
t_tr = np.arange(0, 1.201, 1/30)
x_tr = posicion(t_tr) + generador.normal(0, 0.0015, size=len(t_tr))
y_tr = generador.normal(0, 0.0015, size=len(t_tr))
with open("tracker_movil.txt", "w", encoding="utf-8") as f:
    f.write("Tracker 6.1.4\n")
    f.write("masa A\n")
    f.write("t\tx\ty\n")
    for i, (ti, xi, yi) in enumerate(zip(t_tr, x_tr, y_tr)):
        if i < 2 or i > len(t_tr) - 3:
            f.write(f"{ti:.4f}\tNaN\tNaN\n".replace(".", ","))
        else:
            f.write(f"{ti:.4f}\t{xi:.5f}\t{yi:.5f}\n".replace(".", ","))

# --- Sistema de adquisición: csv con dos líneas de encabezado
t_ad = np.arange(0, 1.201, 0.01)
x_ad = posicion(t_ad) + generador.normal(0, 0.0008, size=len(t_ad))
with open("adquisicion_movil.csv", "w", encoding="utf-8") as f:
    f.write("# Sistema de adquisicion - 100 Hz\n")
    f.write("tiempo(s),posicion(m)\n")
    for ti, xi in zip(t_ad, x_ad):
        f.write(f"{ti:.4f},{xi:.5f}\n")

# --- Photogate: pocos puntos, muy precisos, posiciones fijas
x_pg = np.array([0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70])
t_pg = (-v0_real + np.sqrt(v0_real**2 + 2*a_real*(x_pg - x0_real)))/a_real
t_pg = t_pg + generador.normal(0, 0.0004, size=len(t_pg))
np.savetxt("photogate_movil.txt", np.column_stack([t_pg, x_pg]),
           fmt="%.5f", header="t(s)   x(m)")

print("tres archivos creados")
print(open("tracker_movil.txt", encoding="utf-8").read()[:180])

### 2. Leerlos sin sufrir

`np.loadtxt` no puede con el primero: se traba con el encabezado de dos
líneas, con el tabulador y sobre todo con la coma decimal. La función
`lab.leer_tracker` autodetecta separador y separador decimal, descarta las
líneas de metadatos y recorta los `NaN` de los extremos.

**Punto importante sobre Tracker:** el seguimiento del punto lo hace Tracker
adentro del programa, cuadro por cuadro. Lo que exporta es una tabla
numérica. Acá no hay ningún análisis de imagen que hacer.

In [ ]:
t1, x1, y1 = lab.leer_tracker("tracker_movil.txt")

In [ ]:
nombres, datos = lab.leer_datos("adquisicion_movil.csv")
t2, x2 = datos[:, 0], datos[:, 1]

t3, x3 = np.loadtxt("photogate_movil.txt", unpack=True)
print(f"photogate: {len(t3)} puntos")

### 3. Mirar antes de calcular

La primera cosa que se hace con datos nuevos es un gráfico crudo. Siempre.
Es donde aparecen los cuadros mal trackeados, los saltos de sincronización y
el tramo en que el móvil todavía no arrancó.

In [ ]:
fig, ax = plt.subplots()
ax.plot(t1, x1, "o", ms=4, label=f"Tracker ({len(t1)} pts, 30 Hz)")
ax.plot(t2, x2, ".", ms=3, label=f"Adquisición ({len(t2)} pts, 100 Hz)")
ax.plot(t3, x3, "s", ms=7, mfc="none", label=f"Photogate ({len(t3)} pts)")
ax.set_xlabel("Tiempo (s)")
ax.set_ylabel("Posición (m)")
ax.legend()
plt.show()

Tres muestreos completamente distintos del mismo fenómeno. El photogate da
poquísimos puntos pero muy precisos; el sistema de adquisición da muchos y
buenos; Tracker da una cantidad intermedia con la incerteza más grande,
porque identificar el centro del móvil en una imagen tiene su propio error.

**Frecuencia de muestreo, resolución y aliasing.** Cada instrumento
digitaliza: mide en instantes discretos y con una resolución finita. Si el
fenómeno tiene estructura más rápida que el muestreo, esa estructura no se
pierde de forma inocente: **reaparece disfrazada** de una frecuencia más
baja. Eso es aliasing, y es la razón por la que 30 cuadros por segundo no
sirven para un péndulo rápido.

### 4. Recortar el tramo útil

Rara vez sirve todo el archivo. Se selecciona con máscaras booleanas, que es
la forma idiomática de filtrar en numpy.

In [ ]:
# Nos quedamos con el tramo en que el móvil ya está en movimiento.
util = (t2 > 0.05) & (t2 < 1.15)

print(f"de {len(t2)} puntos me quedo con {util.sum()}")

t2_u, x2_u = t2[util], x2[util]

### 5. Derivar amplifica el ruido

Para sacar la velocidad, la tentación es derivar numéricamente:
$v_i \approx (x_{i+1} - x_{i-1}) / (t_{i+1} - t_{i-1})$, que es lo que hace
`np.gradient`. El problema es que en el numerador va una **diferencia de dos
números parecidos** —donde el ruido no se cancela— y en el denominador un
número chico. El ruido relativo se multiplica.

In [ ]:
v_num = np.gradient(x1, t1)
a_num = np.gradient(v_num, t1)

fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
axes[0].plot(t1, x1, "o", ms=3)
axes[0].set_ylabel("Posición (m)")
axes[1].plot(t1, v_num, "o", ms=3)
axes[1].set_ylabel("Velocidad (m/s)")
axes[2].plot(t1, a_num, "o", ms=3)
axes[2].set_ylabel("Aceleración (m/s²)")
for ax in axes:
    ax.set_xlabel("Tiempo (s)")
plt.show()

print(f"dispersión de la aceleración derivada dos veces: "
      f"{np.std(a_num, ddof=1):.2f} m/s²")
print(f"valor que se quiere medir: {a_real:.3f} m/s²")

La aceleración obtenida derivando dos veces tiene una dispersión **del
orden del valor que se quiere medir**. Es inservible.

La salida correcta es **ajustar el modelo y derivar el modelo**, no los
datos. Si $x(t) = x_0 + v_0 t + \frac12 a t^2$, entonces $a$ es un parámetro
del ajuste y sale con su incerteza, sin amplificar nada. Eso es la Clase 8.

Ojo con el corolario práctico: si Tracker te exporta columnas de velocidad y
aceleración, son diferencias finitas y arrastran exactamente este problema.
No las uses para determinar una aceleración.

### 6. ¿Los tres miden lo mismo?

Ajustamos la parábola a cada serie y comparamos las tres aceleraciones.

In [ ]:
def parabola(t, x0, v0, a):
    return x0 + v0*t + 0.5*a*t**2


series = {
    "Tracker": (t1, x1, 0.0015),
    "Adquisición": (t2_u, x2_u, 0.0008),
    "Photogate": (t3, x3, 0.0004),
}

aceleraciones = {}
for nombre, (t, x, sigma) in series.items():
    p, e, _ = lab.ajustar(parabola, t, x, yerr=np.full(len(t), sigma),
                          verbose=False)
    aceleraciones[nombre] = (p[2], e[2])
    c2r, pv = lab.chi2_reducido(x, parabola(t, *p), np.full(len(t), sigma),
                                3, verbose=False)
    print(f"{nombre:<13} a = {lab.formatear(p[2], e[2], 'm/s²'):<24} "
          f"χ²_ν = {c2r:5.2f}   p = {pv:.3f}")

In [ ]:
nombres_s = list(aceleraciones)
for i in range(len(nombres_s)):
    for j in range(i+1, len(nombres_s)):
        lab.compatibilidad(*aceleraciones[nombres_s[i]],
                           *aceleraciones[nombres_s[j]],
                           etiquetas=(nombres_s[i], nombres_s[j]))
        print()

In [ ]:
valores = np.array([v for v, _ in aceleraciones.values()])
errores = np.array([e for _, e in aceleraciones.values()])

print("Combinación de las tres determinaciones:")
lab.promedio_ponderado(valores, errores)

### 7. Ejercicios

1. Rehacé todo con tus tres archivos reales. Si `lab.leer_datos` no puede
   con alguno, mirá las primeras líneas con
   `print(open("archivo.txt").read()[:300])` y contame qué tiene de raro.
2. ¿Qué pasa con el $\chi^2_\nu$ del ajuste de Tracker si usás como
   incerteza la mitad de la que corresponde? ¿Y el doble? Ésa es la manera
   de detectar barras de error mal estimadas.
3. Filtrá los datos de Tracker quedándote con uno de cada tres puntos y
   rehacé el ajuste. ¿Cuánto empeora la incerteza de $a$? ¿Coincide con lo
   que esperabas por $1/\sqrt{N}$?
4. Recortá el tramo de la adquisición donde el móvil todavía estaba quieto e
   incluilo en el ajuste. Mirá los residuos: ésa es la firma de un tramo que
   no corresponde al modelo.

In [ ]:
# Espacio de trabajo para los ejercicios.

### Para el Informe 3

Se acumula con la Clase 8. Guardá los tres ajustes y la tabla comparativa.